# ViT Linear Probe — Oxford-IIIT Pet Dataset

Trains a frozen ViT-Base backbone with only the classification head unfrozen (Linear Probing).

**Setup:**
- Model: `ViTLinearProbe` (backbone frozen, head trainable)
- Dataset: Oxford-IIIT Pet (37 breed classes)
- Optimizer: Adam
- LR Scheduler: ReduceLROnPlateau (patience=3)
- Early Stopping: patience=5 on validation loss
- Device target: NVIDIA RTX 3070 Ti Laptop GPU

## 1. Imports & Environment Check

In [ ]:
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import os

sys.path.insert(0, "../Models")
sys.path.insert(0, "../DataLoader")
sys.path.insert(0, "../Training")

from DataLoader   import build_dataloaders
from ViTFinetune  import ViTLinearProbe
from TrainingEngine import TrainingEngine

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device}")
if device.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

> **Set `DATASET_ROOT`** to the folder that contains `images/` and `annotations/`.

In [ ]:
TRAINING_TYPE = input()

In [ ]:
DATASET_ROOT = "../../../Dataset/"

BATCH_SIZE = 64 
NUM_EPOCHS = 30
IMAGE_SIZE = 224
VAL_SPLIT = 0.2
NUM_WORKERS = 4
SEED = 42

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

LR_PATIENCE = 3
LR_FACTOR = 0.3
LR_MIN  = 1e-6 

# ── early stopping ────────────────────────────────────────────────────────────
ES_PATIENCE  = 5
LABEL_MODE = "species"

if LABEL_MODE == "breed":
    NUM_CLASSES  = 37

elif LABEL_MODE == "species":
    NUM_CLASSES  = 2


else:
    raise ValueError(f"LABEL_MODE must be 'breed' or 'species', got '{LABEL_MODE}'")

print(f"Label mode : {LABEL_MODE} ({NUM_CLASSES} classes)")

MODEL_NAME = "vit_base_patch16_224"

## 3. DataLoaders

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    dataset_root= DATASET_ROOT,
    val_split = VAL_SPLIT,
    batch_size = BATCH_SIZE,
    one_hot = False,
    image_size = IMAGE_SIZE,
    num_workers = NUM_WORKERS,
    seed = SEED,
)

class LabelSelector:
    def __init__(self, loader, mode):
        self.loader = loader
        self.mode   = mode

    def __len__(self):
        return len(self.loader)

    def __iter__(self):
        for x, (y1, y2) in self.loader:
            yield x, (y2 if self.mode == "breed" else y1)

train_selector = LabelSelector(train_loader, LABEL_MODE)
val_selector   = LabelSelector(val_loader,   LABEL_MODE)
test_selector  = LabelSelector(test_loader,  LABEL_MODE)

print(f"\nBatches — train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

## 4. Model, Loss, Optimizer & Scheduler

In [ ]:
model = ViTLinearProbe(num_classes=NUM_CLASSES, model_name=MODEL_NAME)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable : {trainable_params:>12,}")
print(f"Frozen    : {total_params - trainable_params:>12,}")
print(f"Total     : {total_params:>12,}")

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.Adam(
    model.trainable_params(),
    lr = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode = "min",
    factor = LR_FACTOR,
    patience = LR_PATIENCE,
    min_lr = LR_MIN,
    verbose = True,
)

## 5. Training Engine

In [ ]:
# TrainingEngine uses its own internal scheduler step logic (for AMP safety).
# We pass None here and call scheduler.step(val_loss) ourselves after evaluate().
engine = TrainingEngine(
    network  = model,
    data_iterator = train_selector,
    loss_fn = loss_fn,
    opt = optimizer,
    compute_device = device,
    scheduler = None,
)

## 6. Early-Stopping Helper

In [ ]:
class EarlyStopping:
    """
    Stops training when validation loss has not improved for `patience` epochs.
    Also saves the best model weights to `save_path`.
    """

    def __init__(self, patience: int = 5, save_path: str = "best_model.pt", delta: float = 1e-4):
        self.patience = patience
        self.save_path = save_path
        self.delta = delta
        self.best_loss  = float("inf")
        self.counter = 0
        self.stop  = False

    def __call__(self, val_loss: float, model: nn.Module) -> None:
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter  = 0
            torch.save(model.state_dict(), self.save_path)
            print(f" New best val_loss={val_loss:.4f} — checkpoint saved to '{self.save_path}'")
        else:
            self.counter += 1
            print(f" Early-stop counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.stop = True
                print("Early stopping triggered.")

CHECKPOINT_BASE = "../Checkpoints/"
os.makedirs(CHECKPOINT_BASE, exist_ok=True)

CHECKPOINT_DIR = CHECKPOINT_BASE+MODEL_NAME+"_"+TRAINING_TYPE+"_"+LABEL_MODE
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Checkpoint directory: {os.path.abspath(CHECKPOINT_DIR)}")
early_stopping = EarlyStopping(patience=ES_PATIENCE, save_path=CHECKPOINT_DIR+"/checkpoint.pt")

## 7. Training Loop

In [ ]:
history = {"train_loss": [], "train_acc": [],
           "val_loss":   [], "val_acc":   [], "val_f1": []}

print(f"Starting training for up to {NUM_EPOCHS} epochs ...\n")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n── Epoch {epoch}/{NUM_EPOCHS} ──")

    train_stats = engine.train_one_epoch(epoch_num=epoch, print_freq=50)

    val_stats = engine.evaluate(val_selector, phase_label="Validation")

    history["train_loss"].append(train_stats["epoch_loss"])
    history["train_acc"].append(train_stats["epoch_acc"])
    history["val_loss"].append(val_stats["epoch_loss"])
    history["val_acc"].append(val_stats["epoch_acc"])
    history["val_f1"].append(val_stats["macro_f1"])

    # ── LR scheduler (ReduceLROnPlateau) ──────────────────────────────────────
    scheduler.step(val_stats["epoch_loss"])
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"  Current LR: {current_lr:.2e}")

    # ── early stopping ────────────────────────────────────────────────────────
    early_stopping(val_stats["epoch_loss"], model)
    if early_stopping.stop:
        print(f"\nTraining stopped early at epoch {epoch}.")
        break

print("\n" + "=" * 70)
print("Training complete.")
print(f"Best val_loss : {early_stopping.best_loss:.4f}")
print(f"Best model    : best_vit_linear_probe.pt")

## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(epochs_ran, history["train_loss"], label="Train")
axes[0].plot(epochs_ran, history["val_loss"],   label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(epochs_ran, history["train_acc"], label="Train")
axes[1].plot(epochs_ran, history["val_acc"],   label="Val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True)

# Macro F1
axes[2].plot(epochs_ran, history["val_f1"], label="Val Macro F1", color="green")
axes[2].set_title("Validation Macro F1")
axes[2].set_xlabel("Epoch")
axes[2].legend()
axes[2].grid(True)

plt.suptitle("ViT Linear Probe — Oxford-IIIT Pet", fontsize=13)
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR+"/training_curves.png", dpi=150)
plt.show()
print("Plot saved to training_curves.png")

## 9. Test Evaluation (Best Checkpoint)

In [ ]:
print("Loading best checkpoint ...")
model.load_state_dict(torch.load(CHECKPOINT_DIR+"/checkpoint.pt", map_location=device))
model.to(device)

test_stats = engine.evaluate(test_selector, phase_label="Test")

print("\n── Final Test Results ──────────────────────────────")
print(f"  Loss      : {test_stats['epoch_loss']:.4f}")
print(f"  Accuracy  : {test_stats['epoch_acc']*100:.2f}%")
print(f"  Macro F1  : {test_stats['macro_f1']*100:.2f}%")
print(f"  Macro P   : {test_stats['macro_precision']*100:.2f}%")
print(f"  Macro R   : {test_stats['macro_recall']*100:.2f}%")